# <h1>MultiIndex / advanced indexing</h1>

<p>This section covers <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-hierarchical">indexing with a MultiIndex</a> and <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-index-types">other advanced indexing features</a>.</p>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/indexing.html#indexing">Indexing and Selecting Data</a> for general indexing documentation.</p>

<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>Whether a copy or a reference is returned for a setting operation may
depend on the context.  This is sometimes called <code>chained assignment</code> and should be avoided.
See <a href="https://pandas.pydata.org/docs/user_guide/indexing.html#indexing-view-versus-copy">Returning a View versus Copy</a>.</p>
</div>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-selection">cookbook</a> for some advanced strategies.</p>

## <h2>Hierarchical indexing (MultiIndex)</h2>

<p>Hierarchical / Multi-level indexing is very exciting as it opens the door to some quite sophisticated data analysis and manipulation, especially for working with higher dimensional data.
In essence, it enables you to store and manipulate data with an arbitrary number of dimensions in lower dimensional data structures like <code>Series</code> (1d) and <code>DataFrame</code> (2d).</p>

<p>In this section, we will show what exactly we mean by “hierarchical” indexing and how it integrates with all of the pandas indexing functionality described above and in prior sections.
Later, when discussing <a href="https://pandas.pydata.org/docs/user_guide/groupby.html#groupby">group by</a> and <a href="https://pandas.pydata.org/docs/user_guide/reshaping.html#reshaping">pivoting and reshaping data</a>, we’ll show non-trivial applications to illustrate how it aids in structuring data for analysis.</p>

<p>See the <a href="https://pandas.pydata.org/docs/user_guide/cookbook.html#cookbook-multi-index">cookbook</a> for some advanced strategies.</p>

### <h3>Creating a MultiIndex (hierarchical index) object</h3>

<p>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.html#pandas.MultiIndex" title="pandas.MultiIndex"><code>MultiIndex</code></a> object is the hierarchical analogue of the standard
<a href="https://pandas.pydata.org/docs/reference/api/pandas.Index.html#pandas.Index" title="pandas.Index"><code>Index</code></a> object which typically stores the axis labels in pandas objects.
You can think of <code>MultiIndex</code> as an array of tuples where each tuple is unique.
A <code>MultiIndex</code> can be created from a list of arrays (using
<a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_arrays.html#pandas.MultiIndex.from_arrays" title="pandas.MultiIndex.from_arrays"><code>MultiIndex.from_arrays()</code></a>), an array of tuples (using
<a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_tuples.html#pandas.MultiIndex.from_tuples" title="pandas.MultiIndex.from_tuples"><code>MultiIndex.from_tuples()</code></a>), a crossed set of iterables (using
<a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_product.html#pandas.MultiIndex.from_product" title="pandas.MultiIndex.from_product"><code>MultiIndex.from_product()</code></a>), or a <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html#pandas.DataFrame" title="pandas.DataFrame"><code>DataFrame</code></a> (using <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_frame.html#pandas.MultiIndex.from_frame" title="pandas.MultiIndex.from_frame"><code>MultiIndex.from_frame()</code></a>).
The <code>Index</code> constructor will attempt to return a <code>MultiIndex</code> when it is passed a list of tuples.
The following examples demonstrate different ways to initialize MultiIndexes.</p>

In [65]:
import pandas as pd
import numpy as np

In [66]:
arrays = [
    ["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"],
    ["one", "two", "one", "two", "one", "two", "one", "two"],
]

tuples = list(zip(*arrays))

In [67]:
tuples

[('bar', 'one'),
 ('bar', 'two'),
 ('baz', 'one'),
 ('baz', 'two'),
 ('foo', 'one'),
 ('foo', 'two'),
 ('qux', 'one'),
 ('qux', 'two')]

In [68]:
index  =  pd.MultiIndex.from_tuples(tuples, names = ["first", "second"])

In [69]:
index

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

In [70]:
s = pd.Series(np.random.randn(8), index=index)

In [71]:
s

first  second
bar    one      -1.105108
       two       0.499945
baz    one       0.343884
       two       0.947246
foo    one       1.205753
       two       1.364906
qux    one      -1.248696
       two      -0.653149
dtype: float64

<p>When you want every pairing of the elements in two iterables, it can be easier to use the <a href="../reference/api/pandas.MultiIndex.from_product.html#pandas.MultiIndex.from_product" title="pandas.MultiIndex.from_product"><code>MultiIndex.from_product()</code></a> method:</p>

In [72]:
iterables  =  [["bar", "baz", "foo", "qux" ], ["one", "two"]]

In [73]:
pd.MultiIndex.from_product(iterables, names=["first", "second"])

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('baz', 'one'),
            ('baz', 'two'),
            ('foo', 'one'),
            ('foo', 'two'),
            ('qux', 'one'),
            ('qux', 'two')],
           names=['first', 'second'])

<p>You can also construct a <code>MultiIndex</code> from a <code>DataFrame</code> directly, using the method <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.from_frame.html#pandas.MultiIndex.from_frame" title="pandas.MultiIndex.from_frame"><code>MultiIndex.from_frame()</code></a>.
This is a complementary method to <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.to_frame.html#pandas.MultiIndex.to_frame" title="pandas.MultiIndex.to_frame"><code>MultiIndex.to_frame()</code></a>.</p>

In [74]:
df = pd.DataFrame(
    [["bar", "one"], ["bar", "two"], ["foo", "one"], ["foo", "two"]],
    columns=["first", "second"],
)

In [75]:
pd.MultiIndex.from_frame(df)

MultiIndex([('bar', 'one'),
            ('bar', 'two'),
            ('foo', 'one'),
            ('foo', 'two')],
           names=['first', 'second'])

<p>As a convenience, you can pass a list of arrays directly into <code>Series</code> or <code>DataFrame</code> to construct a <code>MultiIndex</code> automatically:</p>

In [76]:
arrays = [
    np.array(["bar", "bar", "baz", "baz", "foo", "foo", "qux", "qux"]),
    np.array(["one", "two", "one", "two", "one", "two", "one", "two"]),
]

In [77]:
s = pd.Series(np.random.randn(8), index=arrays)

In [78]:
s

bar  one   -1.165227
     two    0.534720
baz  one    1.018028
     two    2.881433
foo  one   -1.262673
     two    0.558456
qux  one    0.239059
     two    1.292513
dtype: float64

In [79]:
df = pd.DataFrame(np.random.randn(8, 4), index=arrays)

In [80]:
df

0         1         2         3
bar one  2.614511  0.230645  0.357318 -0.439801
    two -0.023954  0.166665 -0.235351 -0.790409
baz one  0.393248 -0.263256  0.337445 -0.069656
    two  0.275755  0.319049  1.003482  0.689662
foo one  1.519295 -0.193169 -0.342714 -0.396348
    two  0.130826 -0.242950  0.117690 -0.840041
qux one  0.172591 -1.761324 -1.384485  0.686814
    two -1.274265 -0.571647  1.051582  1.215926

<p>All of the <code>MultiIndex</code> constructors accept a <code>names</code> argument which stores string names for the levels themselves.
If no names are provided, <code>None</code> will be assigned:</p>

In [81]:
df.index.names

FrozenList([None, None])

<p>This index can back any axis of a pandas object, and the number of <strong>levels</strong> of the index is up to you:</p>

In [82]:
df = pd.DataFrame(np.random.randn(3, 8), index=["A", "B", "C"], columns=index)

In [83]:
df

first        bar                 baz                 foo                 qux  \
second       one       two       one       two       one       two       one   
A      -0.413651 -0.176841 -0.471068  0.515305 -0.226300 -0.274601 -0.068815   
B       2.459069 -0.224772  0.564807 -0.502156 -0.139863 -0.260131  1.434197   
C       0.262548 -0.980697 -1.607838  0.842821 -0.896678  1.791216 -0.429403   

first             
second       two  
A      -0.877588  
B      -0.556532  
C       0.422990

In [84]:
pd.DataFrame(np.random.randn(6, 6), index=index[:6], columns=index[:6])

first              bar                 baz                 foo          
second             one       two       one       two       one       two
first second                                                            
bar   one     0.039980  0.416911  0.303351 -1.628269  0.068104 -0.834831
      two     0.573451 -1.847381  0.150801  0.725559 -0.083876 -0.305597
baz   one    -1.409498  0.069958 -1.121757 -1.767819  0.975938  1.543809
      two     0.318407  1.555980  1.415298  0.263746 -0.458930  0.686134
foo   one    -1.010524 -0.508754 -0.093891 -1.644974  0.756623  0.058584
      two    -1.292521 -1.298704  0.119899 -0.328897 -0.878601  0.198544

<p>We’ve “sparsified” the higher levels of the indexes to make the console output a bit easier on the eyes.
Note that how the index is displayed can be controlled using the
<code>multi_sparse</code> option in <code>pandas.set_options()</code>:</p>

In [85]:
with pd.option_context("display.multi_sparse", False):
    df

<p>It’s worth keeping in mind that there’s nothing preventing you from using tuples as atomic labels on an axis:</p>

In [86]:
pd.Series(np.random.randn(8), index=tuples)

,0
"(bar, one)",1.855150
"(bar, two)",0.515372
"(baz, one)",-0.594970
"(baz, two)",-1.322769
"(foo, one)",1.013338
"(foo, two)",0.446878
"(qux, one)",0.289124
"(qux, two)",-1.613921


<p>The reason that the <code>MultiIndex</code> matters is that it can allow you to do grouping, selection, and reshaping operations as we will describe below and in subsequent areas of the documentation.
As you will see in later sections, you can find yourself working with hierarchically-indexed data without creating a <code>MultiIndex</code> explicitly yourself.
However, when loading data from a file, you may wish to generate your own <code>MultiIndex</code> when preparing the data set.</p>

## <h3>Reconstructing the level labels</h3>

<p>The method <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.get_level_values.html#pandas.MultiIndex.get_level_values" title="pandas.MultiIndex.get_level_values"><code>get_level_values()</code></a> will return a vector of the labels for each location at a particular level:</p>

In [87]:
index.get_level_values(0)

Index(['bar', 'bar', 'baz', 'baz', 'foo', 'foo', 'qux', 'qux'], dtype='object', name='first')

In [88]:
index.get_level_values("second")

Index(['one', 'two', 'one', 'two', 'one', 'two', 'one', 'two'], dtype='object', name='second')

## <h3>Basic indexing on axis with MultiIndex</h3>

<p>One of the important features of hierarchical indexing is that you can select data by a “partial” label identifying a subgroup in the data. <strong>Partial</strong> selection “drops” levels of the hierarchical index in the result in a completely analogous way to selecting a column in a regular DataFrame:</p>

In [89]:
df["bar"]

second,one,two
A,-0.413651,-0.176841
B,2.459069,-0.224772
C,0.262548,-0.980697


In [90]:
df["bar", "one"]

,bar
,one
A,-0.413651
B,2.459069
C,0.262548


In [91]:
df["bar"]["one"]

,one
A,-0.413651
B,2.459069
C,0.262548


In [92]:
s["qux"]

,0
one,0.239059
two,1.292513


<p>See <a href="https://pandas.pydata.org/docs/user_guide/advanced.html#advanced-xs">Cross-section with hierarchical index</a> for how to select on a deeper level.</p>

## <h3>Defined levels</h3>

<p>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.html#pandas.MultiIndex" title="pandas.MultiIndex"><code>MultiIndex</code></a> keeps all the defined levels of an index, even if they are not actually used. When slicing an index, you may notice this.
For example:</p>

In [93]:
df.columns.levels  # original MultiIndex

FrozenList([['bar', 'baz', 'foo', 'qux'], ['one', 'two']])

In [94]:
df[["foo","qux"]].columns.levels  # sliced

FrozenList([['bar', 'baz', 'foo', 'qux'], ['one', 'two']])

<p>This is done to avoid a recomputation of the levels in order to make slicing highly performant. If you want to see only the used levels, you can use the <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.get_level_values.html#pandas.MultiIndex.get_level_values" title="pandas.MultiIndex.get_level_values"><code>get_level_values()</code></a> method.</p>

In [95]:
df[["foo", "qux"]].columns.to_numpy()

array([('foo', 'one'), ('foo', 'two'), ('qux', 'one'), ('qux', 'two')],
      dtype=object)

In [96]:
# for a specific level
df[["foo", "qux"]].columns.get_level_values(0)

Index(['foo', 'foo', 'qux', 'qux'], dtype='object', name='first')

<p>To reconstruct the <code>MultiIndex</code> with only the used levels, the <a href="https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.remove_unused_levels.html#pandas.MultiIndex.remove_unused_levels" title="pandas.MultiIndex.remove_unused_levels"><code>remove_unused_levels()</code></a> method may be used.</p>

In [97]:
new_mi = df[["foo", "qux"]].columns.remove_unused_levels()

In [98]:
new_mi.levels

FrozenList([['foo', 'qux'], ['one', 'two']])

## <h3>Data alignment and using <code>reindex</span></code></h3>

<p>Operations between differently-indexed objects having <code>MultiIndex</span></code> on the axes will work as you expect; data alignment will work the same as an Index of tuples:</p>

In [99]:
s + s[:-2]

bar  one   -2.330454
     two    1.069439
baz  one    2.036055
     two    5.762867
foo  one   -2.525346
     two    1.116912
qux  one         NaN
     two         NaN
dtype: float64

In [100]:
s + s[::2]

bar  one   -2.330454
     two         NaN
baz  one    2.036055
     two         NaN
foo  one   -2.525346
     two         NaN
qux  one    0.478118
     two         NaN
dtype: float64

<p>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.reindex.html#pandas.DataFrame.reindex" title="pandas.DataFrame.reindex"><code>reindex()</code></a> method of <code>Series</code>/<code>DataFrames</code> can be called with another <code>MultiIndex</code>, or even a list or array of tuples:</p>

In [101]:
s.reindex(index[:3])

first  second
bar    one      -1.165227
       two       0.534720
baz    one       1.018028
dtype: float64

In [102]:
s.reindex([("foo", "two"), ("bar", "one"), ("qux", "one"), ("baz", "one")])

,,0
foo,two,0.558456
bar,one,-1.165227
qux,one,0.239059
baz,one,1.018028


# <h2>Advanced indexing with hierarchical index</h2>

<p>Syntactically integrating <code>MultiIndex</code> in advanced indexing with <code>.loc</code> is a bit challenging, but we’ve made every effort to do so. In general, MultiIndex keys take the form of tuples. For example, the following works as you would expect:</p>

In [104]:
df = df.T

In [105]:
df

A         B         C
first second                              
bar   one    -0.413651  2.459069  0.262548
      two    -0.176841 -0.224772 -0.980697
baz   one    -0.471068  0.564807 -1.607838
      two     0.515305 -0.502156  0.842821
foo   one    -0.226300 -0.139863 -0.896678
      two    -0.274601 -0.260131  1.791216
qux   one    -0.068815  1.434197 -0.429403
      two    -0.877588 -0.556532  0.422990

In [106]:
df.loc[("bar", "two")]

,bar
,two
A,-0.176841
B,-0.224772
C,-0.980697


<p>Note that <code>df.loc['bar', 'two']</code> would also work in this example, but this shorthand notation can lead to ambiguity in general.</p>

<p>If you also want to index a specific column with <code>.loc</code>, you must use a tuple like this:</p>

In [107]:
df.loc[("bar", "two"), "A"]

np.float64(-0.17684118824718764)

<p>You don’t have to specify all levels of the <code>MultiIndex</span></code> by passing only the first elements of the tuple. For example, you can use “partial” indexing to get all elements with <code>bar</span></code> in the first level as follows:</p>

In [108]:
df.loc["bar"]

,A,B,C
second,,,
one,-0.413651,2.459069,0.262548
two,-0.176841,-0.224772,-0.980697


<p>This is a shortcut for the slightly more verbose notation <code>df.loc[('bar',),]</span></code> (equivalent to <code>df.loc['bar',]</span></code> in this example).</p>

<p>“Partial” slicing also works quite nicely.</p>

In [109]:
df.loc["baz":"foo"]

A         B         C
first second                              
baz   one    -0.471068  0.564807 -1.607838
      two     0.515305 -0.502156  0.842821
foo   one    -0.226300 -0.139863 -0.896678
      two    -0.274601 -0.260131  1.791216

<p>You can slice with a ‘range’ of values, by providing a slice of tuples.</p>

In [110]:
df.loc[("baz", "two"):("qux", "one")]

A         B         C
first second                              
baz   two     0.515305 -0.502156  0.842821
foo   one    -0.226300 -0.139863 -0.896678
      two    -0.274601 -0.260131  1.791216
qux   one    -0.068815  1.434197 -0.429403

In [111]:
df.loc[("baz", "two"):"foo"]

A         B         C
first second                              
baz   two     0.515305 -0.502156  0.842821
foo   one    -0.226300 -0.139863 -0.896678
      two    -0.274601 -0.260131  1.791216

<p>Passing a list of labels or tuples works similar to reindexing:</p>

In [112]:
df.loc[[("bar", "two"), ("qux", "one")]]

,,A,B,C
first,second,,,
bar,two,-0.176841,-0.224772,-0.980697
qux,one,-0.068815,1.434197,-0.429403


<div class="admonition note">
<p class="admonition-title">Note</p>
<p>It is important to note that tuples and lists are not treated identically in pandas when it comes to indexing. Whereas a tuple is interpreted as one multi-level key, a list is used to specify several keys. Or in other words, tuples go horizontally (traversing levels), lists go vertically (scanning levels).</p>
</div>

<p>Importantly, a list of tuples indexes several complete <code>MultiIndex</span></code> keys, whereas a tuple of lists refer to several values within a level:</p>

In [113]:
s = pd.Series(
    [1, 2, 3, 4, 5, 6],
    index=pd.MultiIndex.from_product([["A", "B"], ["c", "d", "e"]]),
)

In [115]:
s.loc[[("A", "c"), ("B", "d")]]  # list of tuples

,,0
A,c,1
B,d,5


In [116]:
s.loc[(["A", "B"], ["c", "d"])]  # tuple of lists

A  c    1
   d    2
B  c    4
   d    5
dtype: int64

## <h3>Using slicers</h3>

<p>You can slice a <code>MultiIndex</span></code> by providing multiple indexers.</p>

<p>You can provide any of the selectors as if you are indexing by label, see <a href="https://pandas.pydata.org/docs/user_guide/indexing.html#indexing-label">Selection by Label</span></a>, including slices, lists of labels, labels, and boolean indexers.</p>

<p>You can use <code>slice(None)</span></code> to select all the contents of <em>that</em> level. You do not need to specify all the
<em>deeper</em> levels, they will be implied as <code>slice(None)</span></code>.</p>

<p>As usual, <strong>both sides</strong> of the slicers are included as this is label indexing.</p>

<div class="admonition warning">
<p class="admonition-title">Warning</p>
<p>You should specify all axes in the <code>.loc</span></code> specifier, meaning the indexer for the <strong>index</strong> and
for the <strong>columns</strong>. There are some ambiguous cases where the passed indexer could be misinterpreted
as indexing <em>both</em> axes, rather than into say the <code>MultiIndex</span></code> for the rows.</p>
<p>You should do this:</p>

<pre><span></span><span class="n">df</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="nb">slice</span><span class="p">(</span><span class="s2">"A1"</span><span class="p">,</span> <span class="s2">"A3"</span><span class="p">),</span> <span class="o">...</span><span class="p">),</span> <span class="p">:]</span>  <span class="c1"># noqa: E999</span>
</pre>

<p>You should <strong>not</strong> do this:</p>

<pre><span></span><span class="n">df</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="nb">slice</span><span class="p">(</span><span class="s2">"A1"</span><span class="p">,</span> <span class="s2">"A3"</span><span class="p">),</span> <span class="o">...</span><span class="p">)]</span>  <span class="c1"># noqa: E999</span>
</pre>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell24"><span></span><span class="gp">In [51]: </span><span class="k">def</span><span class="w"> </span><span class="nf">mklbl</span><span class="p">(</span><span class="n">prefix</span><span class="p">,</span> <span class="n">n</span><span class="p">):</span>
<span class="gp"></span>    <span class="k">return</span> <span class="p">[</span><span class="s2">"</span><span class="si">%s%s</span><span class="s2">"</span> <span class="o">%</span> <span class="p">(</span><span class="n">prefix</span><span class="p">,</span> <span class="n">i</span><span class="p">)</span> <span class="k">for</span> <span class="n">i</span> <span class="ow">in</span> <span class="nb">range</span><span class="p">(</span><span class="n">n</span><span class="p">)]</span>
<span class="gp"></span>

<span class="gp">In [52]: </span><span class="n">miindex</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">MultiIndex</span><span class="o">.</span><span class="n">from_product</span><span class="p">(</span>
<span class="gp"></span>    <span class="p">[</span><span class="n">mklbl</span><span class="p">(</span><span class="s2">"A"</span><span class="p">,</span> <span class="mi">4</span><span class="p">),</span> <span class="n">mklbl</span><span class="p">(</span><span class="s2">"B"</span><span class="p">,</span> <span class="mi">2</span><span class="p">),</span> <span class="n">mklbl</span><span class="p">(</span><span class="s2">"C"</span><span class="p">,</span> <span class="mi">4</span><span class="p">),</span> <span class="n">mklbl</span><span class="p">(</span><span class="s2">"D"</span><span class="p">,</span> <span class="mi">2</span><span class="p">)]</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [53]: </span><span class="n">micolumns</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">MultiIndex</span><span class="o">.</span><span class="n">from_tuples</span><span class="p">(</span>
<span class="gp"></span>    <span class="p">[(</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">),</span> <span class="p">(</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"bar"</span><span class="p">),</span> <span class="p">(</span><span class="s2">"b"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">),</span> <span class="p">(</span><span class="s2">"b"</span><span class="p">,</span> <span class="s2">"bah"</span><span class="p">)],</span> <span class="n">names</span><span class="o">=</span><span class="p">[</span><span class="s2">"lvl0"</span><span class="p">,</span> <span class="s2">"lvl1"</span><span class="p">]</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [54]: </span><span class="n">dfmi</span> <span class="o">=</span> <span class="p">(</span>
<span class="gp"></span>    <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp"></span>        <span class="n">np</span><span class="o">.</span><span class="n">arange</span><span class="p">(</span><span class="nb">len</span><span class="p">(</span><span class="n">miindex</span><span class="p">)</span> <span class="o">*</span> <span class="nb">len</span><span class="p">(</span><span class="n">micolumns</span><span class="p">))</span><span class="o">.</span><span class="n">reshape</span><span class="p">(</span>
<span class="gp"></span>            <span class="p">(</span><span class="nb">len</span><span class="p">(</span><span class="n">miindex</span><span class="p">),</span> <span class="nb">len</span><span class="p">(</span><span class="n">micolumns</span><span class="p">))</span>
<span class="gp"></span>        <span class="p">),</span>
<span class="gp"></span>        <span class="n">index</span><span class="o">=</span><span class="n">miindex</span><span class="p">,</span>
<span class="gp"></span>        <span class="n">columns</span><span class="o">=</span><span class="n">micolumns</span><span class="p">,</span>
<span class="gp"></span>    <span class="p">)</span>
<span class="gp"></span>    <span class="o">.</span><span class="n">sort_index</span><span class="p">()</span>
<span class="gp"></span>    <span class="o">.</span><span class="n">sort_index</span><span class="p">(</span><span class="n">axis</span><span class="o">=</span><span class="mi">1</span><span class="p">)</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [55]: </span><span class="n">dfmi</span>
<span class="gh">Out[55]: </span>
<span class="go">lvl0           a         b     </span>
<span class="go">lvl1         bar  foo  bah  foo</span>
<span class="go">A0 B0 C0 D0    1    0    3    2</span>
<span class="go">         D1    5    4    7    6</span>
<span class="go">      C1 D0    9    8   11   10</span>
<span class="go">         D1   13   12   15   14</span>
<span class="go">      C2 D0   17   16   19   18</span>
<span class="go">...          ...  ...  ...  ...</span>
<span class="go">A3 B1 C1 D1  237  236  239  238</span>
<span class="go">      C2 D0  241  240  243  242</span>
<span class="go">         D1  245  244  247  246</span>
<span class="go">      C3 D0  249  248  251  250</span>
<span class="go">         D1  253  252  255  254</span>

<span class="go">[64 rows x 4 columns]</span>
</pre>
</div>
</div>

<p>Basic MultiIndex slicing using slices, lists, and labels.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell25"><span></span><span class="gp">In [56]: </span><span class="n">dfmi</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="nb">slice</span><span class="p">(</span><span class="s2">"A1"</span><span class="p">,</span> <span class="s2">"A3"</span><span class="p">),</span> <span class="nb">slice</span><span class="p">(</span><span class="kc">None</span><span class="p">),</span> <span class="p">[</span><span class="s2">"C1"</span><span class="p">,</span> <span class="s2">"C3"</span><span class="p">]),</span> <span class="p">:]</span>
<span class="gh">Out[56]: </span>
<span class="go">lvl0           a         b     </span>
<span class="go">lvl1         bar  foo  bah  foo</span>
<span class="go">A1 B0 C1 D0   73   72   75   74</span>
<span class="go">         D1   77   76   79   78</span>
<span class="go">      C3 D0   89   88   91   90</span>
<span class="go">         D1   93   92   95   94</span>
<span class="go">   B1 C1 D0  105  104  107  106</span>
<span class="go">...          ...  ...  ...  ...</span>
<span class="go">A3 B0 C3 D1  221  220  223  222</span>
<span class="go">   B1 C1 D0  233  232  235  234</span>
<span class="go">         D1  237  236  239  238</span>
<span class="go">      C3 D0  249  248  251  250</span>
<span class="go">         D1  253  252  255  254</span>

<span class="go">[24 rows x 4 columns]</span>
</pre>
</div>
</div>

<p>You can use <a href="https://pandas.pydata.org/docs/reference/api/pandas.IndexSlice.html#pandas.IndexSlice" title="pandas.IndexSlice"><code>pandas.IndexSlice</span></code></a> to facilitate a more natural syntax using <code>:</span></code>, rather than using <code>slice(None)</span></code>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell26"><span></span><span class="gp">In [57]: </span><span class="n">idx</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">IndexSlice</span>

<span class="gp">In [58]: </span><span class="n">dfmi</span><span class="o">.</span><span class="n">loc</span><span class="p">[</span><span class="n">idx</span><span class="p">[:,</span> <span class="p">:,</span> <span class="p">[</span><span class="s2">"C1"</span><span class="p">,</span> <span class="s2">"C3"</span><span class="p">]],</span> <span class="n">idx</span><span class="p">[:,</span> <span class="s2">"foo"</span><span class="p">]]</span>
<span class="gh">Out[58]: </span>
<span class="go">lvl0           a    b</span>
<span class="go">lvl1         foo  foo</span>
<span class="go">A0 B0 C1 D0    8   10</span>
<span class="go">         D1   12   14</span>
<span class="go">      C3 D0   24   26</span>
<span class="go">         D1   28   30</span>
<span class="go">   B1 C1 D0   40   42</span>
<span class="go">...          ...  ...</span>
<span class="go">A3 B0 C3 D1  220  222</span>
<span class="go">   B1 C1 D0  232  234</span>
<span class="go">         D1  236  238</span>
<span class="go">      C3 D0  248  250</span>
<span class="go">         D1  252  254</span>

<span class="go">[32 rows x 2 columns]</span>
</pre>
</div>
</div>

<p>It is possible to perform quite complicated selections using this method on multiple axes at the same time.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell27"><span></span><span class="gp">In [59]: </span><span class="n">dfmi</span><span class="o">.</span><span class="n">loc</span><span class="p">[</span><span class="s2">"A1"</span><span class="p">,</span> <span class="p">(</span><span class="nb">slice</span><span class="p">(</span><span class="kc">None</span><span class="p">),</span> <span class="s2">"foo"</span><span class="p">)]</span>
<span class="gh">Out[59]: </span>
<span class="go">lvl0        a    b</span>
<span class="go">lvl1      foo  foo</span>
<span class="go">B0 C0 D0   64   66</span>
<span class="go">      D1   68   70</span>
<span class="go">   C1 D0   72   74</span>
<span class="go">      D1   76   78</span>
<span class="go">   C2 D0   80   82</span>
<span class="go">...       ...  ...</span>
<span class="go">B1 C1 D1  108  110</span>
<span class="go">   C2 D0  112  114</span>
<span class="go">      D1  116  118</span>
<span class="go">   C3 D0  120  122</span>
<span class="go">      D1  124  126</span>

<span class="go">[16 rows x 2 columns]</span>

<span class="gp">In [60]: </span><span class="n">dfmi</span><span class="o">.</span><span class="n">loc</span><span class="p">[</span><span class="n">idx</span><span class="p">[:,</span> <span class="p">:,</span> <span class="p">[</span><span class="s2">"C1"</span><span class="p">,</span> <span class="s2">"C3"</span><span class="p">]],</span> <span class="n">idx</span><span class="p">[:,</span> <span class="s2">"foo"</span><span class="p">]]</span>
<span class="gh">Out[60]: </span>
<span class="go">lvl0           a    b</span>
<span class="go">lvl1         foo  foo</span>
<span class="go">A0 B0 C1 D0    8   10</span>
<span class="go">         D1   12   14</span>
<span class="go">      C3 D0   24   26</span>
<span class="go">         D1   28   30</span>
<span class="go">   B1 C1 D0   40   42</span>
<span class="go">...          ...  ...</span>
<span class="go">A3 B0 C3 D1  220  222</span>
<span class="go">   B1 C1 D0  232  234</span>
<span class="go">         D1  236  238</span>
<span class="go">      C3 D0  248  250</span>
<span class="go">         D1  252  254</span>

<span class="go">[32 rows x 2 columns]</span>
</pre>
</div>
</div>

<p>Using a boolean indexer you can provide selection related to the <em>values</em>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell28"><span></span><span class="gp">In [61]: </span><span class="n">mask</span> <span class="o">=</span> <span class="n">dfmi</span><span class="p">[(</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"foo"</span><span class="p">)]</span> <span class="o">&gt;</span> <span class="mi">200</span>

<span class="gp">In [62]: </span><span class="n">dfmi</span><span class="o">.</span><span class="n">loc</span><span class="p">[</span><span class="n">idx</span><span class="p">[</span><span class="n">mask</span><span class="p">,</span> <span class="p">:,</span> <span class="p">[</span><span class="s2">"C1"</span><span class="p">,</span> <span class="s2">"C3"</span><span class="p">]],</span> <span class="n">idx</span><span class="p">[:,</span> <span class="s2">"foo"</span><span class="p">]]</span>
<span class="gh">Out[62]: </span>
<span class="go">lvl0           a    b</span>
<span class="go">lvl1         foo  foo</span>
<span class="go">A3 B0 C1 D1  204  206</span>
<span class="go">      C3 D0  216  218</span>
<span class="go">         D1  220  222</span>
<span class="go">   B1 C1 D0  232  234</span>
<span class="go">         D1  236  238</span>
<span class="go">      C3 D0  248  250</span>
<span class="go">         D1  252  254</span>
</pre>
</div>
</div>

<p>You can also specify the <code>axis</span></code> argument to <code>.loc</span></code> to interpret the passed slicers on a single axis.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell29"><span></span><span class="gp">In [63]: </span><span class="n">dfmi</span><span class="o">.</span><span class="n">loc</span><span class="p">(</span><span class="n">axis</span><span class="o">=</span><span class="mi">0</span><span class="p">)[:,</span> <span class="p">:,</span> <span class="p">[</span><span class="s2">"C1"</span><span class="p">,</span> <span class="s2">"C3"</span><span class="p">]]</span>
<span class="gh">Out[63]: </span>
<span class="go">lvl0           a         b     </span>
<span class="go">lvl1         bar  foo  bah  foo</span>
<span class="go">A0 B0 C1 D0    9    8   11   10</span>
<span class="go">         D1   13   12   15   14</span>
<span class="go">      C3 D0   25   24   27   26</span>
<span class="go">         D1   29   28   31   30</span>
<span class="go">   B1 C1 D0   41   40   43   42</span>
<span class="go">...          ...  ...  ...  ...</span>
<span class="go">A3 B0 C3 D1  221  220  223  222</span>
<span class="go">   B1 C1 D0  233  232  235  234</span>
<span class="go">         D1  237  236  239  238</span>
<span class="go">      C3 D0  249  248  251  250</span>
<span class="go">         D1  253  252  255  254</span>

<span class="go">[32 rows x 4 columns]</span>
</pre>
</div>
</div>
<p>Furthermore, you can <em>set</em> the values using the following methods.</p>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell30"><span></span><span class="gp">In [64]: </span><span class="n">df2</span> <span class="o">=</span> <span class="n">dfmi</span><span class="o">.</span><span class="n">copy</span><span class="p">()</span>

<span class="gp">In [65]: </span><span class="n">df2</span><span class="o">.</span><span class="n">loc</span><span class="p">(</span><span class="n">axis</span><span class="o">=</span><span class="mi">0</span><span class="p">)[:,</span> <span class="p">:,</span> <span class="p">[</span><span class="s2">"C1"</span><span class="p">,</span> <span class="s2">"C3"</span><span class="p">]]</span> <span class="o">=</span> <span class="o">-</span><span class="mi">10</span>

<span class="gp">In [66]: </span><span class="n">df2</span>
<span class="gh">Out[66]: </span>
<span class="go">lvl0           a         b     </span>
<span class="go">lvl1         bar  foo  bah  foo</span>
<span class="go">A0 B0 C0 D0    1    0    3    2</span>
<span class="go">         D1    5    4    7    6</span>
<span class="go">      C1 D0  -10  -10  -10  -10</span>
<span class="go">         D1  -10  -10  -10  -10</span>
<span class="go">      C2 D0   17   16   19   18</span>
<span class="go">...          ...  ...  ...  ...</span>
<span class="go">A3 B1 C1 D1  -10  -10  -10  -10</span>
<span class="go">      C2 D0  241  240  243  242</span>
<span class="go">         D1  245  244  247  246</span>
<span class="go">      C3 D0  -10  -10  -10  -10</span>
<span class="go">         D1  -10  -10  -10  -10</span>

<span class="go">[64 rows x 4 columns]</span>
</pre>
</div>
</div>

<p>You can use a right-hand-side of an alignable object as well.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell31"><span></span><span class="gp">In [67]: </span><span class="n">df2</span> <span class="o">=</span> <span class="n">dfmi</span><span class="o">.</span><span class="n">copy</span><span class="p">()</span>

<span class="gp">In [68]: </span><span class="n">df2</span><span class="o">.</span><span class="n">loc</span><span class="p">[</span><span class="n">idx</span><span class="p">[:,</span> <span class="p">:,</span> <span class="p">[</span><span class="s2">"C1"</span><span class="p">,</span> <span class="s2">"C3"</span><span class="p">]],</span> <span class="p">:]</span> <span class="o">=</span> <span class="n">df2</span> <span class="o">*</span> <span class="mi">1000</span>

<span class="gp">In [69]: </span><span class="n">df2</span>
<span class="gh">Out[69]: </span>
<span class="go">lvl0              a               b        </span>
<span class="go">lvl1            bar     foo     bah     foo</span>
<span class="go">A0 B0 C0 D0       1       0       3       2</span>
<span class="go">         D1       5       4       7       6</span>
<span class="go">      C1 D0    9000    8000   11000   10000</span>
<span class="go">         D1   13000   12000   15000   14000</span>
<span class="go">      C2 D0      17      16      19      18</span>
<span class="go">...             ...     ...     ...     ...</span>
<span class="go">A3 B1 C1 D1  237000  236000  239000  238000</span>
<span class="go">      C2 D0     241     240     243     242</span>
<span class="go">         D1     245     244     247     246</span>
<span class="go">      C3 D0  249000  248000  251000  250000</span>
<span class="go">         D1  253000  252000  255000  254000</span>

<span class="go">[64 rows x 4 columns]</span>
</pre>
</div>
</div>

## <h3>Cross-section</h3>

<p>The <a href="https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.xs.html#pandas.DataFrame.xs" title="pandas.DataFrame.xs"><code>xs()</span></code></a> method of <code>DataFrame</span></code> additionally takes a level argument to make selecting data at a particular level of a <code>MultiIndex</span></code> easier.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell32"><span></span><span class="gp">In [70]: </span><span class="n">df</span>
<span class="gh">Out[70]: </span>
<span class="go">                     A         B         C</span>
<span class="go">first second                              </span>
<span class="go">bar   one     0.895717  0.410835 -1.413681</span>
<span class="go">      two     0.805244  0.813850  1.607920</span>
<span class="go">baz   one    -1.206412  0.132003  1.024180</span>
<span class="go">      two     2.565646 -0.827317  0.569605</span>
<span class="go">foo   one     1.431256 -0.076467  0.875906</span>
<span class="go">      two     1.340309 -1.187678 -2.211372</span>
<span class="go">qux   one    -1.170299  1.130127  0.974466</span>
<span class="go">      two    -0.226169 -1.436737 -2.006747</span>

<span class="gp">In [71]: </span><span class="n">df</span><span class="o">.</span><span class="n">xs</span><span class="p">(</span><span class="s2">"one"</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="s2">"second"</span><span class="p">)</span>
<span class="gh">Out[71]: </span>
<span class="go">              A         B         C</span>
<span class="go">first                              </span>
<span class="go">bar    0.895717  0.410835 -1.413681</span>
<span class="go">baz   -1.206412  0.132003  1.024180</span>
<span class="go">foo    1.431256 -0.076467  0.875906</span>
<span class="go">qux   -1.170299  1.130127  0.974466</span>
</pre>
</div>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell33"><span></span><span class="go"># using the slicers</span>
<span class="gp">In [72]: </span><span class="n">df</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="nb">slice</span><span class="p">(</span><span class="kc">None</span><span class="p">),</span> <span class="s2">"one"</span><span class="p">),</span> <span class="p">:]</span>
<span class="gh">Out[72]: </span>
<span class="go">                     A         B         C</span>
<span class="go">first second                              </span>
<span class="go">bar   one     0.895717  0.410835 -1.413681</span>
<span class="go">baz   one    -1.206412  0.132003  1.024180</span>
<span class="go">foo   one     1.431256 -0.076467  0.875906</span>
<span class="go">qux   one    -1.170299  1.130127  0.974466</span>
</pre>
</div>
</div>

<p>You can also select on the columns with <code>xs</span></code>, by
providing the axis argument.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell34"><span></span><span class="gp">In [73]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">T</span>

<span class="gp">In [74]: </span><span class="n">df</span><span class="o">.</span><span class="n">xs</span><span class="p">(</span><span class="s2">"one"</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="s2">"second"</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[74]: </span>
<span class="go">first       bar       baz       foo       qux</span>
<span class="go">A      0.895717 -1.206412  1.431256 -1.170299</span>
<span class="go">B      0.410835  0.132003 -0.076467  1.130127</span>
<span class="go">C     -1.413681  1.024180  0.875906  0.974466</span>
</pre>
</div>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell35"><span></span><span class="go"># using the slicers</span>
<span class="gp">In [75]: </span><span class="n">df</span><span class="o">.</span><span class="n">loc</span><span class="p">[:,</span> <span class="p">(</span><span class="nb">slice</span><span class="p">(</span><span class="kc">None</span><span class="p">),</span> <span class="s2">"one"</span><span class="p">)]</span>
<span class="gh">Out[75]: </span>
<span class="go">first        bar       baz       foo       qux</span>
<span class="go">second       one       one       one       one</span>
<span class="go">A       0.895717 -1.206412  1.431256 -1.170299</span>
<span class="go">B       0.410835  0.132003 -0.076467  1.130127</span>
<span class="go">C      -1.413681  1.024180  0.875906  0.974466</span>
</pre>
</div>
</div>

<p><code>xs</span></code> also allows selection with multiple keys.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell36"><span></span><span class="gp">In [76]: </span><span class="n">df</span><span class="o">.</span><span class="n">xs</span><span class="p">((</span><span class="s2">"one"</span><span class="p">,</span> <span class="s2">"bar"</span><span class="p">),</span> <span class="n">level</span><span class="o">=</span><span class="p">(</span><span class="s2">"second"</span><span class="p">,</span> <span class="s2">"first"</span><span class="p">),</span> <span class="n">axis</span><span class="o">=</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[76]: </span>
<span class="go">first        bar</span>
<span class="go">second       one</span>
<span class="go">A       0.895717</span>
<span class="go">B       0.410835</span>
<span class="go">C      -1.413681</span>
</pre>
</div>
</div>
<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell37"><span></span><span class="go"># using the slicers</span>
<span class="gp">In [77]: </span><span class="n">df</span><span class="o">.</span><span class="n">loc</span><span class="p">[:,</span> <span class="p">(</span><span class="s2">"bar"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">)]</span>
<span class="gh">Out[77]: </span>
<span class="go">A    0.895717</span>
<span class="go">B    0.410835</span>
<span class="go">C   -1.413681</span>
<span class="go">Name: (bar, one), dtype: float64</span>
</pre>
</div>
</div>

<p>You can pass <code>drop_level=False</span></code> to <code>xs</span></code> to retain the level that was selected.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell38"><span></span><span class="gp">In [78]: </span><span class="n">df</span><span class="o">.</span><span class="n">xs</span><span class="p">(</span><span class="s2">"one"</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="s2">"second"</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="mi">1</span><span class="p">,</span> <span class="n">drop_level</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="gh">Out[78]: </span>
<span class="go">first        bar       baz       foo       qux</span>
<span class="go">second       one       one       one       one</span>
<span class="go">A       0.895717 -1.206412  1.431256 -1.170299</span>
<span class="go">B       0.410835  0.132003 -0.076467  1.130127</span>
<span class="go">C      -1.413681  1.024180  0.875906  0.974466</span>
</pre>
</div>
</div>

<p>Compare the above with the result using <code>drop_level=True</span></code> (the default value).</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell39"><span></span><span class="gp">In [79]: </span><span class="n">df</span><span class="o">.</span><span class="n">xs</span><span class="p">(</span><span class="s2">"one"</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="s2">"second"</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="mi">1</span><span class="p">,</span> <span class="n">drop_level</span><span class="o">=</span><span class="kc">True</span><span class="p">)</span>
<span class="gh">Out[79]: </span>
<span class="go">first       bar       baz       foo       qux</span>
<span class="go">A      0.895717 -1.206412  1.431256 -1.170299</span>
<span class="go">B      0.410835  0.132003 -0.076467  1.130127</span>
<span class="go">C     -1.413681  1.024180  0.875906  0.974466</span>
</pre>
</div>
</div>

## <h3>Advanced reindexing and alignment</h3>

<p>Using the parameter <code>level</span></code> in the <a href="../reference/api/pandas.DataFrame.reindex.html#pandas.DataFrame.reindex" title="pandas.DataFrame.reindex"><code>reindex()</span></code></a> and
<a href="../reference/api/pandas.DataFrame.align.html#pandas.DataFrame.align" title="pandas.DataFrame.align"><code>align()</span></code></a> methods of pandas objects is useful to broadcast
values across a level. For instance:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell40"><span></span><span class="gp">In [80]: </span><span class="n">midx</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">MultiIndex</span><span class="p">(</span>
<span class="gp"></span>    <span class="n">levels</span><span class="o">=</span><span class="p">[[</span><span class="s2">"zero"</span><span class="p">,</span> <span class="s2">"one"</span><span class="p">],</span> <span class="p">[</span><span class="s2">"x"</span><span class="p">,</span> <span class="s2">"y"</span><span class="p">]],</span> <span class="n">codes</span><span class="o">=</span><span class="p">[[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">],</span> <span class="p">[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">]]</span>
<span class="gp"></span><span class="p">)</span>
<span class="gp"></span>

<span class="gp">In [81]: </span><span class="n">df</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">4</span><span class="p">,</span> <span class="mi">2</span><span class="p">),</span> <span class="n">index</span><span class="o">=</span><span class="n">midx</span><span class="p">)</span>

<span class="gp">In [82]: </span><span class="n">df</span>
<span class="gh">Out[82]: </span>
<span class="go">               0         1</span>
<span class="go">one  y  1.519970 -0.493662</span>
<span class="go">     x  0.600178  0.274230</span>
<span class="go">zero y  0.132885 -0.023688</span>
<span class="go">     x  2.410179  1.450520</span>

<span class="gp">In [83]: </span><span class="n">df2</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">groupby</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span><span class="o">.</span><span class="n">mean</span><span class="p">()</span>

<span class="gp">In [84]: </span><span class="n">df2</span>
<span class="gh">Out[84]: </span>
<span class="go">             0         1</span>
<span class="go">one   1.060074 -0.109716</span>
<span class="go">zero  1.271532  0.713416</span>

<span class="gp">In [85]: </span><span class="n">df2</span><span class="o">.</span><span class="n">reindex</span><span class="p">(</span><span class="n">df</span><span class="o">.</span><span class="n">index</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span>
<span class="gh">Out[85]: </span>
<span class="go">               0         1</span>
<span class="go">one  y  1.060074 -0.109716</span>
<span class="go">     x  1.060074 -0.109716</span>
<span class="go">zero y  1.271532  0.713416</span>
<span class="go">     x  1.271532  0.713416</span>

<span class="go"># aligning</span>
<span class="gp">In [86]: </span><span class="n">df_aligned</span><span class="p">,</span> <span class="n">df2_aligned</span> <span class="o">=</span> <span class="n">df</span><span class="o">.</span><span class="n">align</span><span class="p">(</span><span class="n">df2</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span>

<span class="gp">In [87]: </span><span class="n">df_aligned</span>
<span class="gh">Out[87]: </span>
<span class="go">               0         1</span>
<span class="go">one  y  1.519970 -0.493662</span>
<span class="go">     x  0.600178  0.274230</span>
<span class="go">zero y  0.132885 -0.023688</span>
<span class="go">     x  2.410179  1.450520</span>

<span class="gp">In [88]: </span><span class="n">df2_aligned</span>
<span class="gh">Out[88]: </span>
<span class="go">               0         1</span>
<span class="go">one  y  1.060074 -0.109716</span>
<span class="go">     x  1.060074 -0.109716</span>
<span class="go">zero y  1.271532  0.713416</span>
<span class="go">     x  1.271532  0.713416</span>
</pre>
</div>
</div>

## <h3>Swapping levels with <code>swaplevel</span></code></h3>

<p>The <a href="../reference/api/pandas.MultiIndex.swaplevel.html#pandas.MultiIndex.swaplevel" title="pandas.MultiIndex.swaplevel"><code>swaplevel()</span></code></a> method can switch the order of two levels:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell41"><span></span><span class="gp">In [89]: </span><span class="n">df</span><span class="p">[:</span><span class="mi">5</span><span class="p">]</span>
<span class="gh">Out[89]: </span>
<span class="go">               0         1</span>
<span class="go">one  y  1.519970 -0.493662</span>
<span class="go">     x  0.600178  0.274230</span>
<span class="go">zero y  0.132885 -0.023688</span>
<span class="go">     x  2.410179  1.450520</span>

<span class="gp">In [90]: </span><span class="n">df</span><span class="p">[:</span><span class="mi">5</span><span class="p">]</span><span class="o">.</span><span class="n">swaplevel</span><span class="p">(</span><span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span>
<span class="gh">Out[90]: </span>
<span class="go">               0         1</span>
<span class="go">y one   1.519970 -0.493662</span>
<span class="go">x one   0.600178  0.274230</span>
<span class="go">y zero  0.132885 -0.023688</span>
<span class="go">x zero  2.410179  1.450520</span>
</pre>
</div>
</div>

## <h3>Reordering levels with <code>reorder_levels</code></h3>

<p>The <a href="../reference/api/pandas.MultiIndex.reorder_levels.html#pandas.MultiIndex.reorder_levels" title="pandas.MultiIndex.reorder_levels"><code>reorder_levels()</span></code></a> method generalizes the <code>swaplevel</span></code> method, allowing you to permute the hierarchical index levels in one step:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell42"><span></span><span class="gp">In [91]: </span><span class="n">df</span><span class="p">[:</span><span class="mi">5</span><span class="p">]</span><span class="o">.</span><span class="n">reorder_levels</span><span class="p">([</span><span class="mi">1</span><span class="p">,</span> <span class="mi">0</span><span class="p">],</span> <span class="n">axis</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span>
<span class="gh">Out[91]: </span>
<span class="go">               0         1</span>
<span class="go">y one   1.519970 -0.493662</span>
<span class="go">x one   0.600178  0.274230</span>
<span class="go">y zero  0.132885 -0.023688</span>
<span class="go">x zero  2.410179  1.450520</span>
</pre>
</div>
</div>

## <h3>Renaming names of an <code>Index</code> or <code>MultiIndex</code></h3>

<p>The <a href="../reference/api/pandas.DataFrame.rename.html#pandas.DataFrame.rename" title="pandas.DataFrame.rename"><code>rename()</span></code></a> method is used to rename the labels of a
<code>MultiIndex</span></code>, and is typically used to rename the columns of a <code>DataFrame</span></code>.
The <code>columns</span></code> argument of <code>rename</span></code> allows a dictionary to be specified that includes only the columns you wish to rename.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell43"><span></span><span class="gp">In [92]: </span><span class="n">df</span><span class="o">.</span><span class="n">rename</span><span class="p">(</span><span class="n">columns</span><span class="o">=</span><span class="p">{</span><span class="mi">0</span><span class="p">:</span> <span class="s2">"col0"</span><span class="p">,</span> <span class="mi">1</span><span class="p">:</span> <span class="s2">"col1"</span><span class="p">})</span>
<span class="gh">Out[92]: </span>
<span class="go">            col0      col1</span>
<span class="go">one  y  1.519970 -0.493662</span>
<span class="go">     x  0.600178  0.274230</span>
<span class="go">zero y  0.132885 -0.023688</span>
<span class="go">     x  2.410179  1.450520</span>
</pre>
</div>
</div>

<p>This method can also be used to rename specific labels of the main index of the <code>DataFrame</code>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell44"><span></span><span class="gp">In [93]: </span><span class="n">df</span><span class="o">.</span><span class="n">rename</span><span class="p">(</span><span class="n">index</span><span class="o">=</span><span class="p">{</span><span class="s2">"one"</span><span class="p">:</span> <span class="s2">"two"</span><span class="p">,</span> <span class="s2">"y"</span><span class="p">:</span> <span class="s2">"z"</span><span class="p">})</span>
<span class="gh">Out[93]: </span>
<span class="go">               0         1</span>
<span class="go">two  z  1.519970 -0.493662</span>
<span class="go">     x  0.600178  0.274230</span>
<span class="go">zero z  0.132885 -0.023688</span>
<span class="go">     x  2.410179  1.450520</span>
</pre>
</div>
</div>

<p>The <a href="../reference/api/pandas.DataFrame.rename_axis.html#pandas.DataFrame.rename_axis" title="pandas.DataFrame.rename_axis"><code>rename_axis()</span></code></a> method is used to rename the name of a <code>Index</span></code> or <code>MultiIndex</span></code>. In particular, the names of the levels of a
<code>MultiIndex</span></code> can be specified, which is useful if <code>reset_index()</span></code> is later used to move the values from the <code>MultiIndex</span></code> to a column.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell45"><span></span><span class="gp">In [94]: </span><span class="n">df</span><span class="o">.</span><span class="n">rename_axis</span><span class="p">(</span><span class="n">index</span><span class="o">=</span><span class="p">[</span><span class="s2">"abc"</span><span class="p">,</span> <span class="s2">"def"</span><span class="p">])</span>
<span class="gh">Out[94]: </span>
<span class="go">                 0         1</span>
<span class="go">abc  def                    </span>
<span class="go">one  y    1.519970 -0.493662</span>
<span class="go">     x    0.600178  0.274230</span>
<span class="go">zero y    0.132885 -0.023688</span>
<span class="go">     x    2.410179  1.450520</span>
</pre>
</div>
</div>

<p>Note that the columns of a <code>DataFrame</span></code> are an index, so that using <code>rename_axis</span></code> with the <code>columns</span></code> argument will change the name of that
index.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell46"><span></span><span class="gp">In [95]: </span><span class="n">df</span><span class="o">.</span><span class="n">rename_axis</span><span class="p">(</span><span class="n">columns</span><span class="o">=</span><span class="s2">"Cols"</span><span class="p">)</span><span class="o">.</span><span class="n">columns</span>
<span class="gh">Out[95]: </span><span class="go">RangeIndex(start=0, stop=2, step=1, name='Cols')</span>
</pre>
</div>
</div>

<p>Both <code>rename</span></code> and <code>rename_axis</span></code> support specifying a dictionary, <code>Series</span></code> or a mapping function to map labels/names to new values.</p>

<p>When working with an <code>Index</span></code> object directly, rather than via a <code>DataFrame</span></code>, <a href="../reference/api/pandas.Index.set_names.html#pandas.Index.set_names" title="pandas.Index.set_names"><code>Index.set_names()</span></code></a> can be used to change the names.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell47"><span></span><span class="gp">In [96]: </span><span class="n">mi</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">MultiIndex</span><span class="o">.</span><span class="n">from_product</span><span class="p">([[</span><span class="mi">1</span><span class="p">,</span> <span class="mi">2</span><span class="p">],</span> <span class="p">[</span><span class="s2">"a"</span><span class="p">,</span> <span class="s2">"b"</span><span class="p">]],</span> <span class="n">names</span><span class="o">=</span><span class="p">[</span><span class="s2">"x"</span><span class="p">,</span> <span class="s2">"y"</span><span class="p">])</span>

<span class="gp">In [97]: </span><span class="n">mi</span><span class="o">.</span><span class="n">names</span>
<span class="gh">Out[97]: </span><span class="go">FrozenList(['x', 'y'])</span>

<span class="gp">In [98]: </span><span class="n">mi2</span> <span class="o">=</span> <span class="n">mi</span><span class="o">.</span><span class="n">rename</span><span class="p">(</span><span class="s2">"new name"</span><span class="p">,</span> <span class="n">level</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span>

<span class="gp">In [99]: </span><span class="n">mi2</span>
<span class="gh">Out[99]: </span>
<span class="go">MultiIndex([(1, 'a'),</span>
<span class="go">            (1, 'b'),</span>
<span class="go">            (2, 'a'),</span>
<span class="go">            (2, 'b')],</span>
<span class="go">           names=['new name', 'y'])</span>
</pre>
</div>
</div>

<p>You cannot set the names of the MultiIndex via a level.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell48"><span></span><span class="gp">In [100]: </span><span class="n">mi</span><span class="o">.</span><span class="n">levels</span><span class="p">[</span><span class="mi">0</span><span class="p">]</span><span class="o">.</span><span class="n">name</span> <span class="o">=</span> <span class="s2">"name via level"</span>
<span class="gt">---------------------------------------------------------------------------</span>
<span class="ne">RuntimeError</span><span class="g g-Whitespace">                              </span>Traceback (most recent call last)
<span class="n">Cell</span> <span class="n">In</span><span class="p">[</span><span class="mi">100</span><span class="p">],</span> <span class="n">line</span> <span class="mi">1</span>
<span class="ne">----&gt; </span><span class="mi">1</span> <span class="n">mi</span><span class="o">.</span><span class="n">levels</span><span class="p">[</span><span class="mi">0</span><span class="p">]</span><span class="o">.</span><span class="n">name</span> <span class="o">=</span> <span class="s2">"name via level"</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexes/base.py:1697,</span> in <span class="ni">Index.name</span><span class="nt">(self, value)</span>
<span class="g g-Whitespace">   </span><span class="mi">1693</span> <span class="nd">@name</span><span class="o">.</span><span class="n">setter</span>
<span class="g g-Whitespace">   </span><span class="mi">1694</span> <span class="k">def</span><span class="w"> </span><span class="nf">name</span><span class="p">(</span><span class="bp">self</span><span class="p">,</span> <span class="n">value</span><span class="p">:</span> <span class="n">Hashable</span><span class="p">)</span> <span class="o">-&gt;</span> <span class="kc">None</span><span class="p">:</span>
<span class="g g-Whitespace">   </span><span class="mi">1695</span>     <span class="k">if</span> <span class="bp">self</span><span class="o">.</span><span class="n">_no_setting_name</span><span class="p">:</span>
<span class="g g-Whitespace">   </span><span class="mi">1696</span>         <span class="c1"># Used in MultiIndex.levels to avoid silently ignoring name updates.</span>
<span class="ne">-&gt; </span><span class="mi">1697</span>         <span class="k">raise</span> <span class="ne">RuntimeError</span><span class="p">(</span>
<span class="g g-Whitespace">   </span><span class="mi">1698</span>             <span class="s2">"Cannot set name on a level of a MultiIndex. Use "</span>
<span class="g g-Whitespace">   </span><span class="mi">1699</span>             <span class="s2">"'MultiIndex.set_names' instead."</span>
<span class="g g-Whitespace">   </span><span class="mi">1700</span>         <span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">1701</span>     <span class="n">maybe_extract_name</span><span class="p">(</span><span class="n">value</span><span class="p">,</span> <span class="kc">None</span><span class="p">,</span> <span class="nb">type</span><span class="p">(</span><span class="bp">self</span><span class="p">))</span>
<span class="g g-Whitespace">   </span><span class="mi">1702</span>     <span class="bp">self</span><span class="o">.</span><span class="n">_name</span> <span class="o">=</span> <span class="n">value</span>

<span class="ne">RuntimeError</span>: Cannot set name on a level of a MultiIndex. Use 'MultiIndex.set_names' instead.
</pre>
</div>
</div>

<p>Use <a href="../reference/api/pandas.Index.set_names.html#pandas.Index.set_names" title="pandas.Index.set_names"><code>Index.set_names()</span></code></a> instead.</p>
</section>
</section>

# <h2>Sorting a <code>MultiIndex</code></h2>

<p>For <a class="reference internal" href="../reference/api/pandas.MultiIndex.html#pandas.MultiIndex" title="pandas.MultiIndex"><code class="xref py py-class docutils literal notranslate">MultiIndex</span></code></a>-ed objects to be indexed and sliced effectively, they need to be sorted. As with any index, you can use <a class="reference internal" href="../reference/api/pandas.DataFrame.sort_index.html#pandas.DataFrame.sort_index" title="pandas.DataFrame.sort_index"><code>sort_index()</span></code></a>.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell49"><span></span><span class="gp">In [101]: </span><span class="kn">import</span><span class="w"> </span><span class="nn">random</span>

<span class="gp">In [102]: </span><span class="n">random</span><span class="o">.</span><span class="n">shuffle</span><span class="p">(</span><span class="n">tuples</span><span class="p">)</span>

<span class="gp">In [103]: </span><span class="n">s</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">Series</span><span class="p">(</span><span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">randn</span><span class="p">(</span><span class="mi">8</span><span class="p">),</span> <span class="n">index</span><span class="o">=</span><span class="n">pd</span><span class="o">.</span><span class="n">MultiIndex</span><span class="o">.</span><span class="n">from_tuples</span><span class="p">(</span><span class="n">tuples</span><span class="p">))</span>

<span class="gp">In [104]: </span><span class="n">s</span>
<span class="gh">Out[104]: </span>
<span class="go">baz  one    0.206053</span>
<span class="go">foo  one   -0.251905</span>
<span class="go">baz  two   -2.213588</span>
<span class="go">qux  one    1.063327</span>
<span class="go">foo  two    1.266143</span>
<span class="go">bar  two    0.299368</span>
<span class="go">     one   -0.863838</span>
<span class="go">qux  two    0.408204</span>
<span class="go">dtype: float64</span>

<span class="gp">In [105]: </span><span class="n">s</span><span class="o">.</span><span class="n">sort_index</span><span class="p">()</span>
<span class="gh">Out[105]: </span>
<span class="go">bar  one   -0.863838</span>
<span class="go">     two    0.299368</span>
<span class="go">baz  one    0.206053</span>
<span class="go">     two   -2.213588</span>
<span class="go">foo  one   -0.251905</span>
<span class="go">     two    1.266143</span>
<span class="go">qux  one    1.063327</span>
<span class="go">     two    0.408204</span>
<span class="go">dtype: float64</span>

<span class="gp">In [106]: </span><span class="n">s</span><span class="o">.</span><span class="n">sort_index</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="mi">0</span><span class="p">)</span>
<span class="gh">Out[106]: </span>
<span class="go">bar  one   -0.863838</span>
<span class="go">     two    0.299368</span>
<span class="go">baz  one    0.206053</span>
<span class="go">     two   -2.213588</span>
<span class="go">foo  one   -0.251905</span>
<span class="go">     two    1.266143</span>
<span class="go">qux  one    1.063327</span>
<span class="go">     two    0.408204</span>
<span class="go">dtype: float64</span>

<span class="gp">In [107]: </span><span class="n">s</span><span class="o">.</span><span class="n">sort_index</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[107]: </span>
<span class="go">bar  one   -0.863838</span>
<span class="go">baz  one    0.206053</span>
<span class="go">foo  one   -0.251905</span>
<span class="go">qux  one    1.063327</span>
<span class="go">bar  two    0.299368</span>
<span class="go">baz  two   -2.213588</span>
<span class="go">foo  two    1.266143</span>
<span class="go">qux  two    0.408204</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>

<p id="advanced-sortlevel-byname">You may also pass a level name to <code>sort_index</span></code> if the <code>MultiIndex</span></code> levels
are named.</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell50"><span></span><span class="gp">In [108]: </span><span class="n">s</span><span class="o">.</span><span class="n">index</span> <span class="o">=</span> <span class="n">s</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">set_names</span><span class="p">([</span><span class="s2">"L1"</span><span class="p">,</span> <span class="s2">"L2"</span><span class="p">])</span>

<span class="gp">In [109]: </span><span class="n">s</span><span class="o">.</span><span class="n">sort_index</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="s2">"L1"</span><span class="p">)</span>
<span class="gh">Out[109]: </span>
<span class="go">L1   L2 </span>
<span class="go">bar  one   -0.863838</span>
<span class="go">     two    0.299368</span>
<span class="go">baz  one    0.206053</span>
<span class="go">     two   -2.213588</span>
<span class="go">foo  one   -0.251905</span>
<span class="go">     two    1.266143</span>
<span class="go">qux  one    1.063327</span>
<span class="go">     two    0.408204</span>
<span class="go">dtype: float64</span>

<span class="gp">In [110]: </span><span class="n">s</span><span class="o">.</span><span class="n">sort_index</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="s2">"L2"</span><span class="p">)</span>
<span class="gh">Out[110]: </span>
<span class="go">L1   L2 </span>
<span class="go">bar  one   -0.863838</span>
<span class="go">baz  one    0.206053</span>
<span class="go">foo  one   -0.251905</span>
<span class="go">qux  one    1.063327</span>
<span class="go">bar  two    0.299368</span>
<span class="go">baz  two   -2.213588</span>
<span class="go">foo  two    1.266143</span>
<span class="go">qux  two    0.408204</span>
<span class="go">dtype: float64</span>
</pre>
</div>
</div>

<p>On higher dimensional objects, you can sort any of the other axes by level if they have a <code>MultiIndex</span></code>:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell51"><span></span><span class="gp">In [111]: </span><span class="n">df</span><span class="o">.</span><span class="n">T</span><span class="o">.</span><span class="n">sort_index</span><span class="p">(</span><span class="n">level</span><span class="o">=</span><span class="mi">1</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="mi">1</span><span class="p">)</span>
<span class="gh">Out[111]: </span>
<span class="go">        one      zero       one      zero</span>
<span class="go">          x         x         y         y</span>
<span class="go">0  0.600178  2.410179  1.519970  0.132885</span>
<span class="go">1  0.274230  1.450520 -0.493662 -0.023688</span>
</pre>
</div>
</div>

<p>Indexing will work even if the data are not sorted, but will be rather inefficient (and show a <code>PerformanceWarning</span></code>). It will also
return a copy of the data rather than a view:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell52"><span></span><span class="gp">In [112]: </span><span class="n">dfm</span> <span class="o">=</span> <span class="n">pd</span><span class="o">.</span><span class="n">DataFrame</span><span class="p">(</span>
<span class="gp">   .....: </span>    <span class="p">{</span><span class="s2">"jim"</span><span class="p">:</span> <span class="p">[</span><span class="mi">0</span><span class="p">,</span> <span class="mi">0</span><span class="p">,</span> <span class="mi">1</span><span class="p">,</span> <span class="mi">1</span><span class="p">],</span> <span class="s2">"joe"</span><span class="p">:</span> <span class="p">[</span><span class="s2">"x"</span><span class="p">,</span> <span class="s2">"x"</span><span class="p">,</span> <span class="s2">"z"</span><span class="p">,</span> <span class="s2">"y"</span><span class="p">],</span> <span class="s2">"jolie"</span><span class="p">:</span> <span class="n">np</span><span class="o">.</span><span class="n">random</span><span class="o">.</span><span class="n">rand</span><span class="p">(</span><span class="mi">4</span><span class="p">)}</span>
<span class="gp">   .....: </span><span class="p">)</span>
<span class="gp">   .....: </span>

<span class="gp">In [113]: </span><span class="n">dfm</span> <span class="o">=</span> <span class="n">dfm</span><span class="o">.</span><span class="n">set_index</span><span class="p">([</span><span class="s2">"jim"</span><span class="p">,</span> <span class="s2">"joe"</span><span class="p">])</span>

<span class="gp">In [114]: </span><span class="n">dfm</span>
<span class="gh">Out[114]: </span>
<span class="go">            jolie</span>
<span class="go">jim joe          </span>
<span class="go">0   x    0.490671</span>
<span class="go">    x    0.120248</span>
<span class="go">1   z    0.537020</span>
<span class="go">    y    0.110968</span>

<span class="gp">In [115]: </span><span class="n">dfm</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="mi">1</span><span class="p">,</span> <span class="s1">'z'</span><span class="p">)]</span>
<span class="gh">Out[115]: </span>
<span class="go">           jolie</span>
<span class="go">jim joe         </span>
<span class="go">1   z    0.53702</span>
</pre>
</div>
</div>

<p>Furthermore, if you try to index something that is not fully lexsorted, this can raise:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell53"><span></span><span class="gp">In [116]: </span><span class="n">dfm</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="mi">0</span><span class="p">,</span> <span class="s1">'y'</span><span class="p">):(</span><span class="mi">1</span><span class="p">,</span> <span class="s1">'z'</span><span class="p">)]</span>
<span class="gt">---------------------------------------------------------------------------</span>
<span class="ne">UnsortedIndexError</span><span class="g g-Whitespace">                        </span>Traceback (most recent call last)
<span class="n">Cell</span> <span class="n">In</span><span class="p">[</span><span class="mi">116</span><span class="p">],</span> <span class="n">line</span> <span class="mi">1</span>
<span class="ne">----&gt; </span><span class="mi">1</span> <span class="n">dfm</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="mi">0</span><span class="p">,</span> <span class="s1">'y'</span><span class="p">):(</span><span class="mi">1</span><span class="p">,</span> <span class="s1">'z'</span><span class="p">)]</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexing.py:1191,</span> in <span class="ni">_LocationIndexer.__getitem__</span><span class="nt">(self, key)</span>
<span class="g g-Whitespace">   </span><span class="mi">1189</span> <span class="n">maybe_callable</span> <span class="o">=</span> <span class="n">com</span><span class="o">.</span><span class="n">apply_if_callable</span><span class="p">(</span><span class="n">key</span><span class="p">,</span> <span class="bp">self</span><span class="o">.</span><span class="n">obj</span><span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">1190</span> <span class="n">maybe_callable</span> <span class="o">=</span> <span class="bp">self</span><span class="o">.</span><span class="n">_check_deprecated_callable_usage</span><span class="p">(</span><span class="n">key</span><span class="p">,</span> <span class="n">maybe_callable</span><span class="p">)</span>
<span class="ne">-&gt; </span><span class="mi">1191</span> <span class="k">return</span> <span class="bp">self</span><span class="o">.</span><span class="n">_getitem_axis</span><span class="p">(</span><span class="n">maybe_callable</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="n">axis</span><span class="p">)</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexing.py:1411,</span> in <span class="ni">_LocIndexer._getitem_axis</span><span class="nt">(self, key, axis)</span>
<span class="g g-Whitespace">   </span><span class="mi">1409</span> <span class="k">if</span> <span class="nb">isinstance</span><span class="p">(</span><span class="n">key</span><span class="p">,</span> <span class="nb">slice</span><span class="p">):</span>
<span class="g g-Whitespace">   </span><span class="mi">1410</span>     <span class="bp">self</span><span class="o">.</span><span class="n">_validate_key</span><span class="p">(</span><span class="n">key</span><span class="p">,</span> <span class="n">axis</span><span class="p">)</span>
<span class="ne">-&gt; </span><span class="mi">1411</span>     <span class="k">return</span> <span class="bp">self</span><span class="o">.</span><span class="n">_get_slice_axis</span><span class="p">(</span><span class="n">key</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="n">axis</span><span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">1412</span> <span class="k">elif</span> <span class="n">com</span><span class="o">.</span><span class="n">is_bool_indexer</span><span class="p">(</span><span class="n">key</span><span class="p">):</span>
<span class="g g-Whitespace">   </span><span class="mi">1413</span>     <span class="k">return</span> <span class="bp">self</span><span class="o">.</span><span class="n">_getbool_axis</span><span class="p">(</span><span class="n">key</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="n">axis</span><span class="p">)</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexing.py:1443,</span> in <span class="ni">_LocIndexer._get_slice_axis</span><span class="nt">(self, slice_obj, axis)</span>
<span class="g g-Whitespace">   </span><span class="mi">1440</span>     <span class="k">return</span> <span class="n">obj</span><span class="o">.</span><span class="n">copy</span><span class="p">(</span><span class="n">deep</span><span class="o">=</span><span class="kc">False</span><span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">1442</span> <span class="n">labels</span> <span class="o">=</span> <span class="n">obj</span><span class="o">.</span><span class="n">_get_axis</span><span class="p">(</span><span class="n">axis</span><span class="p">)</span>
<span class="ne">-&gt; </span><span class="mi">1443</span> <span class="n">indexer</span> <span class="o">=</span> <span class="n">labels</span><span class="o">.</span><span class="n">slice_indexer</span><span class="p">(</span><span class="n">slice_obj</span><span class="o">.</span><span class="n">start</span><span class="p">,</span> <span class="n">slice_obj</span><span class="o">.</span><span class="n">stop</span><span class="p">,</span> <span class="n">slice_obj</span><span class="o">.</span><span class="n">step</span><span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">1445</span> <span class="k">if</span> <span class="nb">isinstance</span><span class="p">(</span><span class="n">indexer</span><span class="p">,</span> <span class="nb">slice</span><span class="p">):</span>
<span class="g g-Whitespace">   </span><span class="mi">1446</span>     <span class="k">return</span> <span class="bp">self</span><span class="o">.</span><span class="n">obj</span><span class="o">.</span><span class="n">_slice</span><span class="p">(</span><span class="n">indexer</span><span class="p">,</span> <span class="n">axis</span><span class="o">=</span><span class="n">axis</span><span class="p">)</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexes/base.py:6708,</span> in <span class="ni">Index.slice_indexer</span><span class="nt">(self, start, end, step)</span>
<span class="g g-Whitespace">   </span><span class="mi">6664</span> <span class="k">def</span><span class="w"> </span><span class="nf">slice_indexer</span><span class="p">(</span>
<span class="g g-Whitespace">   </span><span class="mi">6665</span>     <span class="bp">self</span><span class="p">,</span>
<span class="g g-Whitespace">   </span><span class="mi">6666</span>     <span class="n">start</span><span class="p">:</span> <span class="n">Hashable</span> <span class="o">|</span> <span class="kc">None</span> <span class="o">=</span> <span class="kc">None</span><span class="p">,</span>
<span class="g g-Whitespace">   </span><span class="mi">6667</span>     <span class="n">end</span><span class="p">:</span> <span class="n">Hashable</span> <span class="o">|</span> <span class="kc">None</span> <span class="o">=</span> <span class="kc">None</span><span class="p">,</span>
<span class="g g-Whitespace">   </span><span class="mi">6668</span>     <span class="n">step</span><span class="p">:</span> <span class="nb">int</span> <span class="o">|</span> <span class="kc">None</span> <span class="o">=</span> <span class="kc">None</span><span class="p">,</span>
<span class="g g-Whitespace">   </span><span class="mi">6669</span> <span class="p">)</span> <span class="o">-&gt;</span> <span class="nb">slice</span><span class="p">:</span>
<span class="g g-Whitespace">   </span><span class="mi">6670</span><span class="w">     </span><span class="sd">"""</span>
<span class="g g-Whitespace">   </span><span class="mi">6671</span><span class="sd">     Compute the slice indexer for input labels and step.</span>
<span class="g g-Whitespace">   </span><span class="mi">6672</span><span class="sd"> </span>
<span class="sd">   (...)</span>
<span class="g g-Whitespace">   </span><span class="mi">6706</span><span class="sd">     slice(1, 3, None)</span>
<span class="g g-Whitespace">   </span><span class="mi">6707</span><span class="sd">     """</span>
<span class="ne">-&gt; </span><span class="mi">6708</span>     <span class="n">start_slice</span><span class="p">,</span> <span class="n">end_slice</span> <span class="o">=</span> <span class="bp">self</span><span class="o">.</span><span class="n">slice_locs</span><span class="p">(</span><span class="n">start</span><span class="p">,</span> <span class="n">end</span><span class="p">,</span> <span class="n">step</span><span class="o">=</span><span class="n">step</span><span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">6710</span>     <span class="c1"># return a slice</span>
<span class="g g-Whitespace">   </span><span class="mi">6711</span>     <span class="k">if</span> <span class="ow">not</span> <span class="n">is_scalar</span><span class="p">(</span><span class="n">start_slice</span><span class="p">):</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexes/multi.py:2923,</span> in <span class="ni">MultiIndex.slice_locs</span><span class="nt">(self, start, end, step)</span>
<span class="g g-Whitespace">   </span><span class="mi">2871</span><span class="w"> </span><span class="sd">"""</span>
<span class="g g-Whitespace">   </span><span class="mi">2872</span><span class="sd"> For an ordered MultiIndex, compute the slice locations for input</span>
<span class="g g-Whitespace">   </span><span class="mi">2873</span><span class="sd"> labels.</span>
<span class="sd">   (...)</span>
<span class="g g-Whitespace">   </span><span class="mi">2919</span><span class="sd">                       sequence of such.</span>
<span class="g g-Whitespace">   </span><span class="mi">2920</span><span class="sd"> """</span>
<span class="g g-Whitespace">   </span><span class="mi">2921</span> <span class="c1"># This function adds nothing to its parent implementation (the magic</span>
<span class="g g-Whitespace">   </span><span class="mi">2922</span> <span class="c1"># happens in get_slice_bound method), but it adds meaningful doc.</span>
<span class="ne">-&gt; </span><span class="mi">2923</span> <span class="k">return</span> <span class="nb">super</span><span class="p">()</span><span class="o">.</span><span class="n">slice_locs</span><span class="p">(</span><span class="n">start</span><span class="p">,</span> <span class="n">end</span><span class="p">,</span> <span class="n">step</span><span class="p">)</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexes/base.py:6934,</span> in <span class="ni">Index.slice_locs</span><span class="nt">(self, start, end, step)</span>
<span class="g g-Whitespace">   </span><span class="mi">6932</span> <span class="n">start_slice</span> <span class="o">=</span> <span class="kc">None</span>
<span class="g g-Whitespace">   </span><span class="mi">6933</span> <span class="k">if</span> <span class="n">start</span> <span class="ow">is</span> <span class="ow">not</span> <span class="kc">None</span><span class="p">:</span>
<span class="ne">-&gt; </span><span class="mi">6934</span>     <span class="n">start_slice</span> <span class="o">=</span> <span class="bp">self</span><span class="o">.</span><span class="n">get_slice_bound</span><span class="p">(</span><span class="n">start</span><span class="p">,</span> <span class="s2">"left"</span><span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">6935</span> <span class="k">if</span> <span class="n">start_slice</span> <span class="ow">is</span> <span class="kc">None</span><span class="p">:</span>
<span class="g g-Whitespace">   </span><span class="mi">6936</span>     <span class="n">start_slice</span> <span class="o">=</span> <span class="mi">0</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexes/multi.py:2867,</span> in <span class="ni">MultiIndex.get_slice_bound</span><span class="nt">(self, label, side)</span>
<span class="g g-Whitespace">   </span><span class="mi">2865</span> <span class="k">if</span> <span class="ow">not</span> <span class="nb">isinstance</span><span class="p">(</span><span class="n">label</span><span class="p">,</span> <span class="nb">tuple</span><span class="p">):</span>
<span class="g g-Whitespace">   </span><span class="mi">2866</span>     <span class="n">label</span> <span class="o">=</span> <span class="p">(</span><span class="n">label</span><span class="p">,)</span>
<span class="ne">-&gt; </span><span class="mi">2867</span> <span class="k">return</span> <span class="bp">self</span><span class="o">.</span><span class="n">_partial_tup_index</span><span class="p">(</span><span class="n">label</span><span class="p">,</span> <span class="n">side</span><span class="o">=</span><span class="n">side</span><span class="p">)</span>

<span class="nn">File ~/work/pandas/pandas/pandas/core/indexes/multi.py:2927,</span> in <span class="ni">MultiIndex._partial_tup_index</span><span class="nt">(self, tup, side)</span>
<span class="g g-Whitespace">   </span><span class="mi">2925</span> <span class="k">def</span><span class="w"> </span><span class="nf">_partial_tup_index</span><span class="p">(</span><span class="bp">self</span><span class="p">,</span> <span class="n">tup</span><span class="p">:</span> <span class="nb">tuple</span><span class="p">,</span> <span class="n">side</span><span class="p">:</span> <span class="n">Literal</span><span class="p">[</span><span class="s2">"left"</span><span class="p">,</span> <span class="s2">"right"</span><span class="p">]</span> <span class="o">=</span> <span class="s2">"left"</span><span class="p">):</span>
<span class="g g-Whitespace">   </span><span class="mi">2926</span>     <span class="k">if</span> <span class="nb">len</span><span class="p">(</span><span class="n">tup</span><span class="p">)</span> <span class="o">&gt;</span> <span class="bp">self</span><span class="o">.</span><span class="n">_lexsort_depth</span><span class="p">:</span>
<span class="ne">-&gt; </span><span class="mi">2927</span>         <span class="k">raise</span> <span class="n">UnsortedIndexError</span><span class="p">(</span>
<span class="g g-Whitespace">   </span><span class="mi">2928</span>             <span class="sa">f</span><span class="s2">"Key length (</span><span class="si">{</span><span class="nb">len</span><span class="p">(</span><span class="n">tup</span><span class="p">)</span><span class="si">}</span><span class="s2">) was greater than MultiIndex lexsort depth "</span>
<span class="g g-Whitespace">   </span><span class="mi">2929</span>             <span class="sa">f</span><span class="s2">"(</span><span class="si">{</span><span class="bp">self</span><span class="o">.</span><span class="n">_lexsort_depth</span><span class="si">}</span><span class="s2">)"</span>
<span class="g g-Whitespace">   </span><span class="mi">2930</span>         <span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">2932</span>     <span class="n">n</span> <span class="o">=</span> <span class="nb">len</span><span class="p">(</span><span class="n">tup</span><span class="p">)</span>
<span class="g g-Whitespace">   </span><span class="mi">2933</span>     <span class="n">start</span><span class="p">,</span> <span class="n">end</span> <span class="o">=</span> <span class="mi">0</span><span class="p">,</span> <span class="nb">len</span><span class="p">(</span><span class="bp">self</span><span class="p">)</span>

<span class="ne">UnsortedIndexError</span>: 'Key length (2) was greater than MultiIndex lexsort depth (1)'
</pre>
</div>
</div>

<p>The <code>is_monotonic_increasing()</span></code> method on a <code>MultiIndex</span></code> shows if the index is sorted:</p>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell54"><span></span><span class="gp">In [117]: </span><span class="n">dfm</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">is_monotonic_increasing</span>
<span class="gh">Out[117]: </span><span class="go">False</span>
</pre>
</div>
</div>

<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell55"><span></span><span class="gp">In [118]: </span><span class="n">dfm</span> <span class="o">=</span> <span class="n">dfm</span><span class="o">.</span><span class="n">sort_index</span><span class="p">()</span>

<span class="gp">In [119]: </span><span class="n">dfm</span>
<span class="gh">Out[119]: </span>
<span class="go">            jolie</span>
<span class="go">jim joe          </span>
<span class="go">0   x    0.490671</span>
<span class="go">    x    0.120248</span>
<span class="go">1   y    0.110968</span>
<span class="go">    z    0.537020</span>

<span class="gp">In [120]: </span><span class="n">dfm</span><span class="o">.</span><span class="n">index</span><span class="o">.</span><span class="n">is_monotonic_increasing</span>
<span class="gh">Out[120]: </span><span class="go">True</span>
</pre>
</div>
</div>

<p>And now selection works as expected.</p>


<div class="highlight-ipython notranslate"><div class="highlight"><pre id="codecell56"><span></span><span class="gp">In [121]: </span><span class="n">dfm</span><span class="o">.</span><span class="n">loc</span><span class="p">[(</span><span class="mi">0</span><span class="p">,</span> <span class="s2">"y"</span><span class="p">):(</span><span class="mi">1</span><span class="p">,</span> <span class="s2">"z"</span><span class="p">)]</span>
<span class="gh">Out[121]: </span>
<span class="go">            jolie</span>
<span class="go">jim joe          </span>
<span class="go">1   y    0.110968</span>
<span class="go">    z    0.537020</span>
</pre>
</div>
</div>
</section>